In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
def RK4_6delay(func, X0, sets):
    n_x, dt, n_data, window, h, c, A1, A2, A3, A4, A5, A6, window_length, sigma = sets
    func_sets = (n_x, window, h, c, A1, A2, A3, A4, A5, A6, sigma)

    X = np.zeros((n_data, X0.shape[1]))
    X[:window] = X0

    ti = (window - 1) * dt

    for i in range(window - 1, n_data - 1):
        X_seq = np.vstack([
            X[i],                      # 0
            X[i - 1 * window_length],  # 100
            X[i - 2 * window_length],  # 200
            X[i - 3 * window_length],  # 300
            X[i - 4 * window_length],  # 400
            X[i - 5 * window_length],  # 500
        ])
        k1 = func(X_seq, ti, func_sets)

        X_seq_2 = np.vstack([
            X[i] + dt / 2.0 * k1,
            (X[i - 1 * window_length] + X[i - 1 * window_length + 1]) / 2.0,
            (X[i - 2 * window_length] + X[i - 2 * window_length + 1]) / 2.0,
            (X[i - 3 * window_length] + X[i - 3 * window_length + 1]) / 2.0,
            (X[i - 4 * window_length] + X[i - 4 * window_length + 1]) / 2.0,
            (X[i - 5 * window_length] + X[i - 5 * window_length + 1]) / 2.0,
        ])
        k2 = func(X_seq_2, ti + dt / 2.0, func_sets)

        X_seq_3 = np.vstack([
            X[i] + dt / 2.0 * k2,
            (X[i - 1 * window_length] + X[i - 1 * window_length + 1]) / 2.0,
            (X[i - 2 * window_length] + X[i - 2 * window_length + 1]) / 2.0,
            (X[i - 3 * window_length] + X[i - 3 * window_length + 1]) / 2.0,
            (X[i - 4 * window_length] + X[i - 4 * window_length + 1]) / 2.0,
            (X[i - 5 * window_length] + X[i - 5 * window_length + 1]) / 2.0,
        ])
        k3 = func(X_seq_3, ti + dt / 2.0, func_sets)

        X_seq_4 = np.vstack([
            X[i] + dt * k3,
            X[i - 1 * window_length + 1],
            X[i - 2 * window_length + 1],
            X[i - 3 * window_length + 1],
            X[i - 4 * window_length + 1],
            X[i - 5 * window_length + 1],
        ])
        k4 = func(X_seq_4, ti + dt, func_sets)

        X[i + 1] = X[i] + dt / 6.0 * (k1 + 2.0 * k2 + 2.0 * k3 + k4)
        ti += dt

    return X


def coupled_Lorenz_6delay(x, t, sets):
    n_x, window, h, c, A1, A2, A3, A4, A5, A6, sigma = sets

    x_now = x[0]
    r = np.zeros_like(x_now)

    noise = np.random.normal(loc=0, scale=sigma, size=3 * n_x)

    A_list = [A1, A2, A3, A4, A5, A6]

    for k in range(n_x):
        xk = x_now[3 * k + 0]
        yk = x_now[3 * k + 1]
        zk = x_now[3 * k + 2]

        coupling = 0.0
        for m in range(6):
            coupling += np.sum(A_list[m][k] * (x[m, 1::3] - yk))

        r[3 * k + 0] = -10 * (xk - yk + c * coupling) + noise[3 * k + 0]
        r[3 * k + 1] = 28 * (1 + h[k]) * xk - yk - xk * zk + noise[3 * k + 1]
        r[3 * k + 2] = -8 / 3 * zk + xk * yk + noise[3 * k + 2]

    return r

In [ ]:
def gen_data_norm_6time(n_x, A1, A2, A3, A4, A5, A6, n_data, window, window_length, c = 0.3, sigma_dyn = 0, sigma_obs = 0, dt = 0.01):
    h  = 2*(np.random.rand(n_x)-0.5)*0.06
    
    setting = (n_x, dt, n_data, window, h, c, A1, A2, A3, A4, A5, A6, window_length, sigma_dyn)

    random_vector = np.random.rand(3 * n_x)
    x0 = np.tile(random_vector, (window, 1))



#     X = gen_data(setting, x0)
    X = RK4_6delay(coupled_Lorenz_6delay, x0, setting)
    
    q1 = np.percentile(X, 25, interpolation='midpoint', axis=0)
    q2 = np.percentile(X, 50, interpolation='midpoint', axis=0)
    q3 = np.percentile(X, 75, interpolation='midpoint', axis=0)

    X = (X - q2)/(q3 - q1)
    
    if sigma_obs != 0:
        noise_obs = np.random.normal(loc=0, scale=sigma_obs, size=X.shape)
        X += noise_obs
    
    return X